# SDN on ME4OH full-field OFAM3 SST data
MLP built from three linear layers (LSTM removed)

Note: Assumptions as of 30/06/26
- OFAM3 data is stored for Thursdays across the 1979-2014 time period.
- Pending confirmation from the ME4OH team, I have assumed that the data is the full-field SST values generated by the simulator model (OFAM3) averaged across a calendar week

In [ ]:
# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from skimage.metrics import structural_similarity as SSI
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import r2_score

# Local imports
from Data.load_sst_en411_data import crop_df_to_area
from Data.preprocess_sst_ofam3_shred import NA_DATA_PATH
from helper_functions.evaluation import (
    area_weighted_mean,
    get_lats_longs,
    mask_array_by_lat_long,
)
from helper_functions.plotting_functions import plot_compare_sst_recon
from helper_functions.saving import unpack_metadata

np.random.seed(42)

In [ ]:
colours = [ "#ffb000", "#dc267f", "#1589e8", "#631ff3", "#fe5100", "#000000"]  # IBM colourblind palette  green: "#0d894b"

## Set up

In [ ]:
gru_exp_index = 5
lags = 52
sdn_exp_index = 4
anomaly_ep_index = 6
lstm_exp_index_5 = 3  # 1 for 52 lags; 3 for 2 week lags
lstm_exp_index_1059 = 2  # 2 for 52 lags, 3 for 2 week lags

In [ ]:
# Load data
num_sensors = 5
lats, longs = get_lats_longs(NA_DATA_PATH)
sensor_locations_5, sensor_coords_5, test_lag_dates_5, bulk_mean = unpack_metadata(f"Reconstructions/Static/Exp{anomaly_ep_index}/metadata_n{num_sensors}_l{lags}.json")

anomaly_test_recons_5 = np.load(f"Reconstructions/Static/Exp{anomaly_ep_index}/recons_n{num_sensors}_l{lags}.npy")
actual_test_recons_5 = anomaly_test_recons_5 + bulk_mean
anomaly_ground_truth_5 = np.load(f"Reconstructions/Static/Exp{anomaly_ep_index}/truth_n{num_sensors}_l{lags}.npy")

gru_test_recons_5 = np.load(f"Reconstructions/Static/Exp{gru_exp_index}/recons_n{num_sensors}_l{lags}.npy")
lstm_test_recons_5 = np.load(f"Reconstructions/Static/Exp{lstm_exp_index_5}/recons_n{num_sensors}_l{lags}.npy")
sdn_test_recons_5 = np.load(f"Reconstructions/Static/Exp{sdn_exp_index}/recons_n{num_sensors}_l{lags}.npy")

# Check truth and sensor locs still equal for anomaly data
sdn_ground_truth_5 = np.load(f"Reconstructions/Static/Exp{sdn_exp_index}/truth_n{num_sensors}_l{lags}.npy")
sdn_sensor_locations_5, sdn_sensor_coords_5, _, sdn_test_lag_dates_5 = unpack_metadata(f"Reconstructions/Static/Exp{sdn_exp_index}/metadata_n{num_sensors}_l{lags}.json")

In [ ]:
# Load data
num_sensors = 1059
lats, longs = get_lats_longs(NA_DATA_PATH)
sensor_locations_1059, sensor_coords_1059, test_lag_dates_1059 = unpack_metadata(f"Reconstructions/Static/Exp{anomaly_ep_index}/metadata_n{num_sensors}_l{lags}.json")

anomaly_test_recons_1059 = np.load(f"Reconstructions/Static/Exp{anomaly_ep_index}/recons_n{num_sensors}_l{lags}.npy")
actual_test_recons_1059 = anomaly_test_recons_1059 + bulk_mean
anomaly_ground_truth_1059 = np.load(f"Reconstructions/Static/Exp{anomaly_ep_index}/truth_n{num_sensors}_l{lags}.npy")

gru_test_recons_1059 = np.load(f"Reconstructions/Static/Exp{gru_exp_index}/recons_n{num_sensors}_l{lags}.npy")
sdn_test_recons_1059 = np.load(f"Reconstructions/Static/Exp{sdn_exp_index}/recons_n{num_sensors}_l{lags}.npy")
lstm_test_recons_1059 = np.load(f"Reconstructions/Static/Exp{lstm_exp_index_1059}/recons_n{num_sensors}_l{lags}.npy")

# Check truth and sensor locs still equal for anomaly data
sdn_ground_truth_1059 = np.load(f"Reconstructions/Static/Exp{sdn_exp_index}/truth_n{num_sensors}_l{lags}.npy")
sdn_sensor_locations_1059, sdn_sensor_coords_1059, _, sdn_test_lag_dates_1059 = unpack_metadata(f"Reconstructions/Static/Exp{sdn_exp_index}/metadata_n{num_sensors}_l{lags}.json")


In [ ]:
# Add on bulk mean value
assert np.unique(anomaly_ground_truth_5 == anomaly_ground_truth_1059)[0]
ANOMALY_GROUND_TRUTH = anomaly_ground_truth_5

In [ ]:
# Check did run all models against identical test sets?
# Check sensors in same place

# Anomaly vs SDN & GRU (checked in Exp 5)
assert np.unique(test_lag_dates_5 == sdn_test_lag_dates_5)[0]
assert np.unique(test_lag_dates_1059 == sdn_test_lag_dates_1059)[0]

# Because have to add bulk mean back on, there can be a difference in ground truths up to 0.001
assert np.unique(np.isclose(ANOMALY_GROUND_TRUTH + bulk_mean, sdn_ground_truth_5, rtol=1e-3))[0]
assert np.unique(np.isclose(ANOMALY_GROUND_TRUTH + bulk_mean, sdn_ground_truth_1059, rtol=1e-3))[0]
assert np.unique(sdn_ground_truth_5 == sdn_ground_truth_1059)[0]
GROUND_TRUTH = sdn_ground_truth_5

assert np.unique(sensor_coords_5 == sdn_sensor_coords_5)[0]
assert np.unique(sensor_coords_1059 == sdn_sensor_coords_1059)[0]

# Verify truth is the same
assert np.unique(test_lag_dates_5 == test_lag_dates_1059)[0]
TEST_LAG_DATES = test_lag_dates_5

## Evaluation

In [ ]:
i = 3

In [ ]:
for (test_anomaly, test_gru, test_sdn, truth, coords) in [(actual_test_recons_5, gru_test_recons_5, sdn_test_recons_5, GROUND_TRUTH, sensor_coords_5), 
                                          (actual_test_recons_1059, gru_test_recons_1059, sdn_test_recons_1059, GROUND_TRUTH, sensor_coords_1059)]:
    gru_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_gru[i]
    })
    anomaly_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_anomaly[i]
    })
    sdn_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_sdn[i]
    })
    true_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": truth[i]
    })
    date = TEST_LAG_DATES[i]
    print(f"Date: {date}")
    plot_compare_sst_recon(sdn_df, true_df, area="NA", sensor_coords=coords, title=f"SDN reconstruction, Date: {date}")
    plot_compare_sst_recon(gru_df, true_df, area="NA", sensor_coords=coords, title=f"SHRED(GRU) reconstruction, Date: {date}")
    plot_compare_sst_recon(anomaly_df, true_df, area="NA", sensor_coords=coords, title=f"SHRED(GRU) anomaly reconstruction, Date: {date}")

In [ ]:
for (test_anomaly, truth, coords) in [(anomaly_test_recons_5, ANOMALY_GROUND_TRUTH, sensor_coords_5), 
                                          (anomaly_test_recons_1059, ANOMALY_GROUND_TRUTH, sensor_coords_1059)]:
    anomaly_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_anomaly[i]
    })
    true_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": truth[i]
    })
    date = TEST_LAG_DATES[i]
    print(f"Date: {date}")
    plot_compare_sst_recon(anomaly_df, true_df, area="NA",
                           sensor_coords=coords, cmap="viridis",
                           title=f"SHRED(GRU) anomaly reconstruction, Date: {date}")
    plot_compare_sst_recon(anomaly_df, true_df, area="NA", cmap="viridis",
                           title=f"SHRED(GRU) anomaly reconstruction, Date: {date}")

### Metrics
Structural Similarity Index: [[1]](https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.structural_similarity) [[2]](https://ieeexplore.ieee.org/document/4775883)



#### NA basin

In [ ]:
print("Calculation using actual SST:")
for (num_sensors, test) in [(5, anomaly_test_recons_5+bulk_mean), (1059, anomaly_test_recons_1059+bulk_mean)]:
    print("NUM SENSORS: ", num_sensors, "TESTSET LAGS: ", lags)
    mse = MSE(y_pred=test, y_true=GROUND_TRUTH)
    rmse = np.sqrt(mse)
    mae = MAE(y_pred=test, y_true=GROUND_TRUTH)
    nmse = mse/np.mean(GROUND_TRUTH)**2
    r2 = r2_score(y_pred=test, y_true=GROUND_TRUTH)
    print(f"RMSE:\t{rmse:.3f}")
    print(f"MSE:\t{mse:.3f}")
    print(f"NMSE:\t{nmse:.3f}")
    print(f"MAE:\t{mae:.3f}")
    print(f"R2:\t{r2:.3f}")

    ssi = np.mean([SSI(GROUND_TRUTH[i], test[i], 
                   data_range=(max(test[i].max(), GROUND_TRUTH[i].max()) - min(test[i].min(), GROUND_TRUTH[i].min())))
                   for i in range(len(test))])
    print(f"Average SSI:\t{ssi:.3f}")
    break

In [ ]:
# print("Calculation using anomalies:")  # These match!!! :D
# for (num_sensors, test) in [(5, anomaly_test_recons_5), (1059, anomaly_test_recons_1059)]:
#     print("NUM SENSORS: ", num_sensors, "TESTSET LAGS: ", lags)
#     rmse = np.sqrt(MSE(y_pred=test, y_true=ANOMALY_GROUND_TRUTH))
#     mae = MAE(y_pred=test, y_true=ANOMALY_GROUND_TRUTH)
#     r2 = r2_score(y_pred=test, y_true=ANOMALY_GROUND_TRUTH)
#     print(f"RMSE:\t{rmse:.3f}")
#     print(f"MAE:\t{mae:.3f}")
#     print(f"R2:\t{r2:.3f}")

#     ssi = np.mean([SSI(ANOMALY_GROUND_TRUTH[i], test[i], 
#                    data_range=(max(test[i].max(), ANOMALY_GROUND_TRUTH[i].max()) - min(test[i].min(), ANOMALY_GROUND_TRUTH[i].min())))
#                    for i in range(len(test))])
#     print(f"Average SSI:\t{ssi:.3f}")

#### Gulf-Stream region

In [ ]:
for (num_sensors, test) in [(5, actual_test_recons_5), (1059, actual_test_recons_1059)]:
    print("NUM SENSORS:", num_sensors, "TESTSET LAGS: ", lags)
    GS_test_data = mask_array_by_lat_long(test, lats, longs)
    GS_truth_data = mask_array_by_lat_long(GROUND_TRUTH, lats, longs)
    GS_test_data = [d[~np.isnan(d)] for d in GS_test_data]
    GS_truth_data = [d[~np.isnan(d)] for d in GS_truth_data]

    rmse = np.sqrt(MSE(y_pred=GS_test_data, y_true=GS_truth_data))
    mae = MAE(y_pred=GS_test_data, y_true=GS_truth_data)
    r2 = r2_score(y_pred=GS_test_data, y_true=GS_truth_data)
    print(f"RMSE:\t{rmse:.3f}")
    print(f"MAE:\t{mae:.3f}")
    print(f"R2:\t{r2:.3f}")

    ssi = np.mean([SSI(GS_truth_data[i], GS_test_data[i],
                       data_range=(max(test[i].max(), GROUND_TRUTH[i].max()) - min(test[i].min(), GROUND_TRUTH[i].min())))
                                          for i in range(len(test))])
    print(f"Average SSI:\t{ssi:.3f}")

### Regional integral of SST (area weighted mean)

In [ ]:
truth_sst_mean, _ = area_weighted_mean(GROUND_TRUTH, lats, longs)
anomaly_truth_sst_mean, _ = area_weighted_mean(ANOMALY_GROUND_TRUTH, lats, longs)
# same truth for all data with matching sequence lengths (see beginning asserts)

gru_test_sst_mean_5, _ = area_weighted_mean(gru_test_recons_5, lats, longs)
actual_test_sst_mean_5, _ = area_weighted_mean(actual_test_recons_5, lats, longs)
anomaly_test_sst_mean_5, _ = area_weighted_mean(anomaly_test_recons_5, lats, longs)
sdn_test_sst_mean_5, _ = area_weighted_mean(sdn_test_recons_5, lats, longs)
lstm_test_sst_mean_5, _ = area_weighted_mean(lstm_test_recons_5, lats, longs)

gru_test_sst_mean_1059, _ = area_weighted_mean(gru_test_recons_1059, lats, longs)
actual_test_sst_mean_1059, _ = area_weighted_mean(actual_test_recons_1059, lats, longs)
anomaly_test_sst_mean_1059, _ = area_weighted_mean(anomaly_test_recons_1059, lats, longs)
sdn_test_sst_mean_1059, _ = area_weighted_mean(sdn_test_recons_1059, lats, longs)
lstm_test_sst_mean_1059, _ = area_weighted_mean(lstm_test_recons_1059, lats, longs)

In [ ]:
sst_mean_df = pd.DataFrame({
    "Date": TEST_LAG_DATES,
    # "SHRED (GRU) - 5 sensors": gru_test_sst_mean_5,
    "SHRED (GRU) - 1059 sensors": gru_test_sst_mean_1059,
    "SHRED (LSTM) - 1059 sensors": lstm_test_sst_mean_1059,
    "SDN - 1059 sensors": sdn_test_sst_mean_1059,
    "SHRED (GRU) Anomaly - 5 sensors": actual_test_sst_mean_5,
    "SHRED (GRU) Anomaly - 1059 sensors": actual_test_sst_mean_1059,
    # "SDN - 5 sensors": sdn_test_sst_mean_5,
    "Truth": truth_sst_mean
})
# Smooth over 1 year = 52 weeks
sst_rolling_df = pd.DataFrame({
    "Date": TEST_LAG_DATES,
    # "SHRED (GRU) - 5 sensors": sst_mean_df["SHRED (GRU) - 5 sensors"].rolling(52).mean(),
    "SHRED (GRU) - 1059 sensors": sst_mean_df["SHRED (GRU) - 1059 sensors"].rolling(52).mean(),
    "SHRED (LSTM) - 1059 sensors": sst_mean_df["SHRED (LSTM) - 1059 sensors"].rolling(52).mean(),
    "SDN - 1059 sensors": sst_mean_df["SDN - 1059 sensors"].rolling(52).mean(),
    "SHRED (GRU) Anomaly - 5 sensors": sst_mean_df["SHRED (GRU) Anomaly - 5 sensors"].rolling(52).mean(),
    "SHRED (GRU) Anomaly - 1059 sensors": sst_mean_df["SHRED (GRU) Anomaly - 1059 sensors"].rolling(52).mean(),
    # "SDN - 5 sensors": sst_mean_df["SDN - 5 sensors"].rolling(52).mean(),
    
    "Truth": sst_mean_df.Truth.rolling(52).mean()
})
sst_mean_df = sst_mean_df.set_index("Date")
sst_rolling_df = sst_rolling_df.set_index("Date")

In [ ]:
plt.figure(figsize=(8,8))
sst_mean_df[["SHRED (GRU) - 1059 sensors", "SHRED (LSTM) - 1059 sensors", "SDN - 1059 sensors",
             "SHRED (GRU) Anomaly - 5 sensors", "SHRED (GRU) Anomaly - 1059 sensors",
             "Truth"]].plot(colormap=ListedColormap(colours), ls='--')
plt.xticks(rotation=90)
plt.ylabel("Mean SST ($\\degree$C)")
plt.legend(bbox_to_anchor=(1,1))
plt.show()

In [ ]:
plt.figure(figsize=(8,8))
sst_rolling_df[["SHRED (GRU) - 1059 sensors", "SHRED (LSTM) - 1059 sensors", "SDN - 1059 sensors",
             "SHRED (GRU) Anomaly - 5 sensors", "SHRED (GRU) Anomaly - 1059 sensors",
             "Truth"]].plot(colormap=ListedColormap(colours), ls='--')
plt.xticks(rotation=90)
plt.ylabel("Smoothed mean SST ($\\degree$C)")
plt.legend(bbox_to_anchor=(1,1))
plt.show()

### For anomalies

### Difference between test truth and reconstructions 

In [ ]:
gru_difference_5 = gru_test_recons_5 - GROUND_TRUTH
lstm_difference_5 = lstm_test_recons_5 - GROUND_TRUTH
sdn_difference_5 = sdn_test_recons_5 - GROUND_TRUTH
actual_difference_5 = actual_test_recons_5 - GROUND_TRUTH

gru_difference_1059 = gru_test_recons_1059 - GROUND_TRUTH
lstm_difference_1059 = lstm_test_recons_1059 - GROUND_TRUTH
sdn_difference_1059 = sdn_test_recons_1059 - GROUND_TRUTH
actual_difference_1059 = actual_test_recons_1059 - GROUND_TRUTH

#### Difference plots

In [ ]:
plot_index = 3

#### Actual SST

In [ ]:
for (num_sensors, model_str, diff, test) in ([5, "SHRED (GRU) Anomaly", actual_difference_5, actual_test_recons_5],
                                             [1059, "SHRED (GRU) Anomaly", actual_difference_1059, actual_test_recons_1059]):
    for i in [plot_index]:  #range(0, len(test_recons), 10):  # np.random.choice(range(len(test_recons)), 1):
        diff_df = pd.DataFrame({
                "Latitude": lats,
                "Longitude": longs,
                "SST": diff[i]
            })
        recon_df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": longs,
            "SST": test[i]
        })
        true_df = pd.read_csv(f"{NA_DATA_PATH}{TEST_LAG_DATES[i]}.csv")
        # plot_sst_map(diff_df, title=f"Difference map for {test_dates[i]}", min_sst=-5, max_sst=5, area="NA", cmap='bwr')
        print(f"{model_str} - {num_sensors} sensors for date: {TEST_LAG_DATES[i]}")
        plot_compare_sst_recon(recon_df, true_df, diff_data=diff_df, area="NA")  #, sensor_coords=sensor_coords)

        gs_recon_df = crop_df_to_area(recon_df, "GS", "Latitude")
        gs_true_df = crop_df_to_area(true_df, "GS", "Latitude")
        gs_diff_df = crop_df_to_area(diff_df, "GS", "Latitude")
        
        plot_compare_sst_recon(gs_recon_df, gs_true_df, diff_data=gs_diff_df, area='GS')

#### Anomaly differences

In [ ]:
for (num_sensors, model_str, test) in ([5, "SHRED (GRU) Anomaly", anomaly_test_recons_5],
                                       [1059, "SHRED (GRU) Anomaly", anomaly_test_recons_1059]):
    diff = test - ANOMALY_GROUND_TRUTH
    for i in [plot_index]:  #range(0, len(test_recons), 10):  # np.random.choice(range(len(test_recons)), 1):
        diff_df = pd.DataFrame({
                "Latitude": lats,
                "Longitude": longs,
                "SST": diff[i]
            })
        recon_df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": longs,
            "SST": test[i]
        })
        true_df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": longs,
            "SST": ANOMALY_GROUND_TRUTH[i]
        })
        # plot_sst_map(diff_df, title=f"Difference map for {test_dates[i]}", min_sst=-5, max_sst=5, area="NA", cmap='bwr')
        print(f"{model_str} - {num_sensors} sensors for date: {TEST_LAG_DATES[i]}")
        plot_compare_sst_recon(recon_df, true_df, diff_data=diff_df, area="NA", cmap="viridis")  #, sensor_coords=sensor_coords)

        gs_recon_df = crop_df_to_area(recon_df, "GS", "Latitude")
        gs_true_df = crop_df_to_area(true_df, "GS", "Latitude")
        gs_diff_df = crop_df_to_area(diff_df, "GS", "Latitude")
        
        plot_compare_sst_recon(gs_recon_df, gs_true_df, diff_data=gs_diff_df, area='GS', cmap="viridis")

#### Zonally integrated average difference 
i.e. Average difference between reconstruction and truth across all years;
plotted against latitude (no weighting by area needed, because each square in latitude band has same area)

OR could scale by area by doing lat_band/total_area????

In [ ]:
avg_all_time_lat_avg_diffs = []
labels = []
for (num_sensors, model_str, difference) in [(5, "SHRED (GRU)", gru_difference_5),
                                              (1059, "SHRED (GRU)", gru_difference_1059),
                                              (5, "SHRED (LSTM)", lstm_difference_5),
                                              (1059, "SHRED (LSTM)", lstm_difference_1059),
                                              (5, "SDN", sdn_difference_5),
                                              (1059, "SDN", sdn_difference_1059),
                                              (5, "SHRED (GRU) Anomaly", actual_difference_5),
                                              (1059, "SHRED (GRU) Anomaly", actual_difference_1059),
                                              ]:
    
    # For each reconstruction at a set date, find the average difference at each latitude
    latitude_avg_diffs = np.array([pd.Series(diff_at_time).groupby(lats).mean().to_numpy() for diff_at_time in difference])

    # For each latitude, calculate the mean difference over all time
    avg_all_time_lat_avg_diffs.append(np.array([np.mean(latitude_avg_diffs.T[j]) for j in range(latitude_avg_diffs.shape[1])]))
    labels.append(f"{model_str} - {num_sensors} sensors")

In [ ]:
colour_index = 0
for i in range(len(avg_all_time_lat_avg_diffs)):
    if "Anomaly" in labels[i]:
        plt.plot(avg_all_time_lat_avg_diffs[i], np.unique(lats), c=colours[colour_index], label=labels[i])
        colour_index += 1
    elif "1059" in labels[i]:
        plt.plot(avg_all_time_lat_avg_diffs[i], np.unique(lats), c=colours[colour_index], label=labels[i])
        colour_index += 1
plt.vlines(0, 0, 60, colors=['k'], linestyles=['--'])
plt.xlabel('Average difference (from the truth) ($\\degree$C)')
plt.ylabel('Latitude')
plt.legend(bbox_to_anchor=(1,1))

##### Anomaly difference

In [ ]:
# avg_all_time_lat_avg_diffs = []
# labels = []
for (num_sensors, model_str, test) in [(5, "SHRED (GRU) Anomaly (from anomaly)", anomaly_test_recons_5), (1059, "SHRED (GRU) Anomaly (from anomaly)", anomaly_test_recons_1059)]:
    difference = test - ANOMALY_GROUND_TRUTH

    # For each reconstruction at a set date, find the average difference at each latitude
    latitude_avg_diffs = np.array([pd.Series(diff_at_time).groupby(lats).mean().to_numpy() for diff_at_time in difference])

    # For each latitude, calculate the mean difference over all time
    avg_all_time_lat_avg_diffs.append(np.array([np.mean(latitude_avg_diffs.T[j]) for j in range(latitude_avg_diffs.shape[1])]))
    labels.append(f"{model_str} - {num_sensors} sensors")

In [ ]:
colour_index = 0
for i in range(len(avg_all_time_lat_avg_diffs)):
    if "from" in labels[i]:
        plt.plot(avg_all_time_lat_avg_diffs[i], np.unique(lats), c=colours[colour_index], label=labels[i], ls=':')
        colour_index += 1
    elif "Anomaly" in labels[i]:
            plt.plot(avg_all_time_lat_avg_diffs[i], np.unique(lats), c=colours[colour_index], label=labels[i], ls='-.')
            colour_index += 1
plt.vlines(0, 0, 60, colors=['k'], linestyles=['--'])
plt.xlabel('Average difference (from the truth) ($\\degree$C)')
plt.ylabel('Latitude')
plt.legend(bbox_to_anchor=(1,1))

#### Average difference between reconstruction and truth across whole NA region - by year

For 1059 sensor GRU model:

In [ ]:
avg_difference = [np.mean(d) for d in actual_difference_1059]
avg_all_time_difference = np.mean(avg_difference)
print(avg_all_time_difference)
plt.figure(figsize=(8,6))
plt.stem(TEST_LAG_DATES, avg_difference, markerfmt='.')
plt.ylabel('Average difference ($\\degree$C)')
plt.xlabel('Reconstruction date')

In [ ]:
# For Gulf Stream only:
GS_data = mask_array_by_lat_long(actual_difference_1059, lats, longs)
GS_data = [d[~np.isnan(d)] for d in GS_data]

In [ ]:
avg_difference = [np.mean(d) for d in GS_data]
avg_all_time_difference = np.mean(avg_difference)
print(avg_all_time_difference)
plt.figure(figsize=(8,6))
plt.stem(TEST_LAG_DATES, avg_difference, markerfmt='.')
plt.ylabel('Average difference ($\\degree$C)')
plt.xlabel('Reconstruction date')

### Understanding sequences in anomaly space

In [ ]:
import random

from sklearn.preprocessing import MinMaxScaler

from Data.preprocess_sst_ofam3_shred import (
    convert_to_anomaly,
    create_shred_sequences,
    load_files,
    split_ordered_data,
)

In [ ]:
# Load and reformat data
data, dates, lats, longs = load_files()
train_data, val_data, test_data, test_dates = split_ordered_data(data, dates)

train_data, val_data, test_data, bulk_mean = convert_to_anomaly((train_data, val_data, test_data))

# Normalise data
sc = MinMaxScaler()
sc = sc.fit(train_data)  # Computes min and max from training data only (prevent data leakage)
test_transformed = sc.transform(test_data)

In [ ]:
# Build sequences
sensor_locs = sensor_locations_5
num_sensors = len(sensor_locs)

norm_test_dataset = create_shred_sequences(test_transformed, sensor_locs, lags=lags)
anom_test_dataset = create_shred_sequences(test_data, sensor_locs, lags=lags)

for data_name, test_dataset in {"Normalised SST": norm_test_dataset, "Anomaly SST": anom_test_dataset}.items():
    plot_index = 3
    plot_data = test_dataset[plot_index]
    # random_colours = ["#"+''.join([random.choice('0123456789ABCDEF') for j in range(6)])
    #             for i in range(num_sensors)]

    for i in range(num_sensors):
        # plot_data[0] -> test sequences
        # plot_data[0].T -> for each sensor, timeseries of SST values across test dataset
        plt.plot(plot_data[0].T[i], c=colours[i])
    plt.ylabel(data_name)
    plt.xlabel("Week index")
    plt.grid()
    plt.xlim(0, 52)
    plt.show()